# 📊 Data Mining Assignment — Data Lake Preprocessing with Apache Spark

**Objective:** Build a simple Data Lake (Bronze → Silver) in Google Colab using PySpark.  
**Dataset:** `sales.csv`  
**Environment:** Google Colab + PySpark  

---

## 1. Data Lake Structure

Create the folder structure and place the raw `sales.csv` in the **Bronze** layer (unchanged).

In [ ]:
import os

# Create the Data Lake folder structure
os.makedirs('data_lake/bronze', exist_ok=True)
os.makedirs('data_lake/silver/preprocessed_sales', exist_ok=True)

print("✅ Data Lake structure created:")
for root, dirs, files in os.walk('data_lake'):
    level = root.replace('data_lake', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{sub_indent}{file}')

In [ ]:
# Upload sales.csv — this cell opens a file-upload dialog
from google.colab import files

uploaded = files.upload()  # Select sales.csv from your computer

In [ ]:
import shutil

# Copy the uploaded file into the Bronze layer (keeping it unchanged)
shutil.copy('sales.csv', 'data_lake/bronze/sales.csv')
print("✅ sales.csv copied to data_lake/bronze/sales.csv")
print(f"   File size: {os.path.getsize('data_lake/bronze/sales.csv')} bytes")

---
## 2. Load and Inspect with Spark

Install PySpark, create a SparkSession, and load the Bronze data.

In [ ]:
# Install PySpark
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession

# Create SparkSession with LEGACY time parser to handle mixed date formats gracefully
spark = SparkSession.builder \
    .appName("DataLakeAssignment") \
    .master("local[*]") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

print("✅ SparkSession created")
print(f"   Spark version: {spark.version}")

In [ ]:
# Read sales.csv from the Bronze layer
df_bronze = spark.read.csv(
    'data_lake/bronze/sales.csv',
    header=True,
    inferSchema=True
)

print("✅ Bronze data loaded successfully")

### 2.1 Display Schema, Sample Rows, and Record Count

In [ ]:
# Display the schema
print("📋 Schema:")
df_bronze.printSchema()

In [ ]:
# Show first 10 rows
print("📋 First 10 rows:")
df_bronze.show(10, truncate=False)

In [ ]:
# Total record count
total_records = df_bronze.count()
print(f"📊 Total records in Bronze: {total_records}")

### 2.2 Identify Data Quality Issues

In [ ]:
from pyspark.sql import functions as F

# --- 1. Duplicate Rows ---
total_rows = df_bronze.count()
distinct_rows = df_bronze.distinct().count()
duplicate_count = total_rows - distinct_rows
print(f"🔍 DUPLICATES:  {duplicate_count} exact duplicate rows found")
print(f"   (Total rows: {total_rows}, Distinct rows: {distinct_rows})")

In [ ]:
# --- 2. Null / Missing Values ---
print("🔍 NULL VALUES per column:")
null_counts = df_bronze.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_bronze.columns]
)
null_counts.show(truncate=False)

In [ ]:
# --- 3. Text-Format Inconsistencies (Capitalization & Extra Spaces) ---
print("🔍 TEXT INCONSISTENCIES in 'customer_name':")
df_bronze.select('customer_name') \
    .filter(
        # Names with leading/trailing spaces
        (F.col('customer_name') != F.trim(F.col('customer_name'))) |
        # Names that are ALL UPPER
        (F.col('customer_name') == F.upper(F.col('customer_name'))) |
        # Names that are all lower
        (F.col('customer_name') == F.lower(F.col('customer_name')))
    ).show(truncate=False)

In [ ]:
# --- 4. Invalid Numerical Values ---
print("🔍 INVALID NUMERICAL VALUES:")

# Quantity should be > 0
print("\n  Rows with quantity <= 0:")
df_bronze.filter(F.col('quantity') <= 0).select(
    'order_id', 'customer_name', 'product', 'quantity'
).show(truncate=False)

# Unit price should be > 0
print("  Rows with unit_price <= 0:")
df_bronze.filter(F.col('unit_price') <= 0).select(
    'order_id', 'customer_name', 'product', 'unit_price'
).show(truncate=False)

# Discount should be between 0 and 100
print("  Rows with discount_percentage outside [0, 100]:")
df_bronze.filter(
    (F.col('discount_percentage') < 0) | (F.col('discount_percentage') > 100)
).select(
    'order_id', 'customer_name', 'product', 'discount_percentage'
).show(truncate=False)

In [ ]:
# --- 5. Inconsistent / Invalid Dates ---
print("🔍 INCONSISTENT / INVALID DATES:")

# The standard format is yyyy-MM-dd. Let's find rows that don't match.
df_bronze.filter(
    ~F.col('order_date').rlike(r'^\d{4}-\d{2}-\d{2}$')
).select(
    'order_id', 'customer_name', 'order_date'
).show(truncate=False)

**Summary of issues found:**

| # | Issue | Examples |
|---|-------|----------|
| 1 | Exact duplicate rows | order_id 1002 appears twice; order_id 1004 duplicates 1001 |
| 2 | Null / missing values | `customer_name` is null (order 1013), `quantity` is null (order 1023) |
| 3 | Capitalization inconsistencies | `bob smith`, `CAROL WHITE`, `XAVIER KING`, `grace lee` |
| 4 | Extra leading/trailing spaces | `"  Eve Davis  "`, `"  diana carter  "` |
| 5 | Negative/zero quantity | order 1005 (qty = -1), order 1037 (qty = 0) |
| 6 | Negative unit_price | order 1045 (unit_price = -120.00) |
| 7 | Invalid discount_percentage | order 1012 (discount = 150, exceeds 100%) |
| 8 | Inconsistent date formats | `2024/01/21` (slashes), `21-01-2024` (dd-MM-yyyy), `not-a-date` |

---
## 3. Basic Preprocessing — Silver Layer

Apply the six preprocessing steps while preserving as many useful records as possible.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# Start from the raw Bronze DataFrame
df = df_bronze
print(f"Starting row count: {df.count()}")

### Step 1 — Remove Exact Duplicate Rows

In [ ]:
before = df.count()
df = df.dropDuplicates()
after = df.count()
print(f"✅ Step 1 — Removed {before - after} exact duplicate rows")
print(f"   Rows: {before} → {after}")

### Step 2 — Handle Missing Values

In [ ]:
# Fill missing customer_name with "Unknown"
df = df.fillna({'customer_name': 'Unknown'})

# Fill missing quantity with the median quantity (rounded)
median_qty = int(df.filter(F.col('quantity').isNotNull())
                   .approxQuantile('quantity', [0.5], 0.01)[0])
print(f"   Median quantity used for imputation: {median_qty}")
df = df.fillna({'quantity': median_qty})

# Verify no nulls remain in key columns
remaining_nulls = df.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
)
print("✅ Step 2 — Missing values handled")
remaining_nulls.show(truncate=False)

### Step 3 — Trim Extra Spaces & Standardize Capitalization

In [ ]:
# Trim and title-case all string/text columns
text_columns = ['customer_name', 'product', 'category', 'region']

for col_name in text_columns:
    df = df.withColumn(col_name, F.initcap(F.trim(F.col(col_name))))

print("✅ Step 3 — Trimmed spaces & standardized capitalization (Title Case)")
df.select('customer_name').distinct().orderBy('customer_name').show(50, truncate=False)

### Step 4 — Validate Numerical Fields

In [ ]:
before = df.count()

# 4a. Quantity: set invalid values (<=0) to null to preserve the row
df = df.withColumn(
    'quantity',
    F.when(F.col('quantity') <= 0, None).otherwise(F.col('quantity'))
)

# 4b. Unit price: set invalid values (<=0) to null
df = df.withColumn(
    'unit_price',
    F.when(F.col('unit_price') <= 0, None).otherwise(F.col('unit_price'))
)

# 4c. Discount percentage: values > 100 or < 0 are clearly invalid → set to null
df = df.withColumn(
    'discount_percentage',
    F.when(
        (F.col('discount_percentage') < 0) | (F.col('discount_percentage') > 100),
        None
    ).otherwise(F.col('discount_percentage'))
)

after = df.count()
print("✅ Step 4 — Validated numerical fields")
print("   • quantity <= 0       → set to null")
print("   • unit_price <= 0     → set to null")
print("   • discount outside [0,100] → set to null")
print(f"   Rows preserved: {after} (no rows dropped — values nullified instead)")

### Step 5 — Convert Dates to a Consistent Format

In [ ]:
from datetime import datetime
from pyspark.sql.types import DateType

# Define a Python UDF to safely parse multiple date formats
# This avoids the DateTimeException that Spark 3.x throws on invalid dates
def parse_date(date_str):
    if date_str is None:
        return None
    formats = ['%Y-%m-%d', '%Y/%m/%d', '%d-%m-%Y']
    for fmt in formats:
        try:
            return datetime.strptime(date_str.strip(), fmt).date()
        except (ValueError, AttributeError):
            continue
    return None  # Unparseable → null

parse_date_udf = F.udf(parse_date, DateType())

# Apply the UDF to parse dates safely
df = df.withColumn('order_date_parsed', parse_date_udf(F.col('order_date')))

# Show which rows could NOT be parsed (truly invalid dates)
invalid_dates = df.filter(F.col('order_date_parsed').isNull())
print(f"⚠️  Rows with unparseable dates: {invalid_dates.count()}")
invalid_dates.select('order_id', 'order_date').show(truncate=False)

# Replace original column with the parsed date
df = df.drop('order_date').withColumnRenamed('order_date_parsed', 'order_date')

print("✅ Step 5 — Dates converted to consistent yyyy-MM-dd format")
print("   Rows with truly invalid dates kept (order_date set to null)")

### Step 6 — Preserve Useful Records

We intentionally kept all rows; invalid values were nullified rather than removing entire rows.  
This preserves partial data (e.g., a row with a bad date still has valid product/customer info).

In [ ]:
print("✅ Step 6 — All useful records preserved")
print(f"   Final row count after preprocessing: {df.count()}")
print("   (Only exact duplicates were removed; invalid values were nullified, not dropped)")

### Save to Silver Layer

In [ ]:
# Save as CSV inside the Silver folder
df.coalesce(1).write.csv(
    'data_lake/silver/preprocessed_sales',
    header=True,
    mode='overwrite'
)

print("✅ Preprocessed data saved to data_lake/silver/preprocessed_sales/")

In [ ]:
# Also save a clean single CSV file for submission (sales_silver.csv)
import glob

# Find the part file Spark created
part_files = glob.glob('data_lake/silver/preprocessed_sales/part-*.csv')
if part_files:
    shutil.copy(part_files[0], 'data_lake/silver/sales_silver.csv')
    print("✅ Single file saved: data_lake/silver/sales_silver.csv")
    print(f"   Size: {os.path.getsize('data_lake/silver/sales_silver.csv')} bytes")

---
## 4. Verification

Verify the preprocessing by comparing row counts and reading back the Silver data.

In [ ]:
# Row count comparison
bronze_count = df_bronze.count()
silver_count = df.count()

print("📊 ROW COUNT COMPARISON")
print(f"   Bronze (raw):  {bronze_count} rows")
print(f"   Silver (clean): {silver_count} rows")
print(f"   Rows removed:  {bronze_count - silver_count} (duplicates only)")

In [ ]:
# Read back the Silver data with Spark and display
df_silver = spark.read.csv(
    'data_lake/silver/preprocessed_sales',
    header=True,
    inferSchema=True
)

print("📋 Silver data — Schema:")
df_silver.printSchema()

print("\n📋 Silver data — First 10 rows:")
df_silver.show(10, truncate=False)

In [ ]:
# Final summary of preprocessing operations
print("="*65)
print("      PREPROCESSING OPERATIONS SUMMARY")
print("="*65)
print("""
1. REMOVED EXACT DUPLICATES
   → dropDuplicates() removed identical rows.

2. HANDLED MISSING VALUES
   → Null customer_name filled with 'Unknown'.
   → Null quantity filled with the median value.

3. TRIMMED SPACES & STANDARDIZED CAPITALIZATION
   → trim() removed leading/trailing whitespace.
   → initcap() applied Title Case to all text fields
     (customer_name, product, category, region).

4. VALIDATED NUMERICAL FIELDS
   → quantity <= 0       → set to null (preserving the row).
   → unit_price <= 0     → set to null.
   → discount_percentage outside [0, 100] → set to null.

5. CONVERTED DATES TO CONSISTENT FORMAT
   → Parsed yyyy-MM-dd, yyyy/MM/dd, and dd-MM-yyyy
     formats into a single DateType column using a Python UDF.
   → Unparseable values (e.g. 'not-a-date') → null.

6. PRESERVED USEFUL RECORDS
   → No rows were dropped for having invalid values;
     only exact duplicates were removed.
   → Invalid individual field values were nullified
     so partial data remains available.
""")
print("="*65)

In [ ]:
# Download the silver CSV for submission
from google.colab import files
files.download('data_lake/silver/sales_silver.csv')